In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
spark=SparkSession.builder\
       .appName("Scenario_2")\
       .master("local[*]")\
       .config("spark.python.worker.reuse", "false") \
        .config("spark.executor.heartbeatInterval", "60s") \
        .config("spark.network.timeout", "120s") \
        .config("spark.hadoop.io.native.lib.available", "false") \
        .config("spark.sql.shuffle.partitions", "50") \
        .config("spark.sql.adaptive.enabled", "true") \
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
        .getOrCreate()

In [7]:
df1=spark.read.format("parquet")\
         .option("inferschema",True)\
         .option("header",True)\
         .load("E:/Big_data/Ansh_lamba/Post/Post_3/pyspark_scenarios/dataset_generation/data/dataset2_user_events/events/v1/")
df1.show()

+-----------+--------+----------+----------+-------------------+------------------+----------------+-----------+-------+-------+-----------+
|   event_id| user_id|session_id|event_type|    event_timestamp|         page_name|duration_seconds|device_type|     os|country|app_version|
+-----------+--------+----------+----------+-------------------+------------------+----------------+-----------+-------+-------+-----------+
|E0000666708|U0000803| S00269219|     click|2024-01-01 00:00:08|           profile|            NULL|     mobile|    Mac|     US|        3.5|
|E0000429699|U0020459| S00498965|    logout|2024-01-01 00:00:11|order_confirmation|            NULL|     mobile|    iOS|     US|        4.0|
|E0000209181|U0007420| S00189644| page_view|2024-01-01 00:00:16|          checkout|           252.0|    desktop|Android| Brazil|        4.1|
|E0000112732|U0018428| S00206923|  purchase|2024-01-01 00:00:16|     category_page|           451.0|    desktop|Android|  India|        4.1|
|E0000550706|

In [8]:
df1.groupBy('user_id').agg(count('event_timestamp').alias('event_count')).orderBy(col('user_id')).show()

+--------+-----------+
| user_id|event_count|
+--------+-----------+
|U0000001|        147|
|U0000002|        131|
|U0000003|        170|
|U0000004|        146|
|U0000005|        166|
|U0000006|        142|
|U0000007|        156|
|U0000008|        155|
|U0000009|        154|
|U0000010|        158|
|U0000011|        143|
|U0000012|        139|
|U0000013|        156|
|U0000014|        149|
|U0000015|        160|
|U0000016|        152|
|U0000017|        137|
|U0000018|        155|
|U0000019|        150|
|U0000020|        142|
+--------+-----------+
only showing top 20 rows



In [9]:
df4=df1.withColumn('previous_timestamp',lag('event_timestamp',1)\
                   .over(Window.partitionBy('user_id')\
                   .orderBy('event_timestamp')))\
    .withColumn('gap_seconds',unix_timestamp(col('event_timestamp'))-unix_timestamp(col('previous_timestamp')))

df5=df4.filter(col('gap_seconds')<=2)\
    .select('user_id','event_timestamp','previous_timestamp','gap_seconds')\
    .orderBy(col('gap_seconds').desc())


In [10]:
df5.groupBy('user_id')\
    .agg(count('gap_seconds').alias('total_count'))\
    .filter(col('total_count')>10).orderBy('user_id').show()

+--------+-----------+
| user_id|total_count|
+--------+-----------+
|U0000014|        144|
|U0000019|        145|
|U0000056|        141|
|U0000071|        143|
|U0000107|        119|
|U0000117|        152|
|U0000213|        147|
|U0000236|        140|
|U0000257|        149|
|U0000306|        160|
|U0000317|        150|
|U0000377|        134|
|U0000484|        168|
|U0000639|        150|
|U0000690|        127|
|U0000776|        123|
|U0000801|        147|
|U0000813|        155|
|U0000820|        150|
|U0000870|        144|
+--------+-----------+
only showing top 20 rows

